# Construcción de los datasets limpios (ICDAS y segmentación)
**Inputs (Add Input):** `leonardoaranguiz/odontify-manifiestos` (con los CSV limpios), `leonardoaranguiz/odontify-clean-icdas-dataset-v2` y `odontify-clean-v2`. CPU, sin internet.

**Salida:** `/kaggle/working/icdas_crops_curated/` y `/kaggle/working/fdi_segmentation_curated/`. Al terminar: *Output → New Dataset* para cada carpeta.

In [ ]:
import pandas as pd, os, glob, shutil, json, hashlib
IN='/kaggle/input'
def find(name):
    h=glob.glob(f'{IN}/**/{name}',recursive=True); assert h, name; return h[0]
ic=pd.read_csv(find('icdas_manifest.csv')); mp=pd.read_csv(find('map_icdas_privado.csv'))
sg=pd.read_csv(find('segmentation_manifest.csv')); ch=pd.read_csv(find('children.csv'))
print(ic.shape, sg.shape, ch.shape)

## 1. Recortes ICDAS

In [ ]:
O1='/kaggle/working/icdas_crops_curated'; os.makedirs(f'{O1}/crops',exist_ok=True)
idx={}
for dp,_,fs in os.walk(IN):
    for f in fs:
        p=os.path.join(dp,f); idx.setdefault('/'.join(p.split('/')[-2:]),[]).append(p)
print('archivos indexados',len(idx))
faltan=[]
for r in mp.itertuples():
    h=[p for p in idx.get(r.file_v2,[]) if 'odontify-manifiestos' not in p]
    if not h: faltan.append(r.file_v2); continue
    shutil.copy2(h[0], f'{O1}/{r.crop_file}')
print('copiados',len(mp)-len(faltan),'faltan',len(faltan),faltan[:5])
assert not faltan
ic.to_csv(f'{O1}/icdas_manifest.csv',index=False)
ch[ch.in_icdas].drop(columns=['fold_seg','pipeline_pool']).to_csv(f'{O1}/children.csv',index=False)
print(pd.crosstab(ic.fold,ic.micdas,margins=True))

## 2. Segmentación FDI

In [ ]:
O2='/kaggle/working/fdi_segmentation_curated'; os.makedirs(f'{O2}/images',exist_ok=True); os.makedirs(f'{O2}/masks',exist_ok=True)
seg_root=os.path.dirname(find('labelmap.txt'))
print('raíz corpus segmentación:',seg_root, os.listdir(seg_root)[:10])
faltan=[]
for r in sg.itertuples():
    for rel in (r.image_file, r.mask_file):
        p=os.path.join(seg_root,rel)
        if not os.path.exists(p): faltan.append(rel); continue
        shutil.copy2(p, os.path.join(O2,rel))
print('faltan',len(faltan),faltan[:5]); assert not faltan
shutil.copy2(find('labelmap.txt'), f'{O2}/labelmap.txt')
sg.to_csv(f'{O2}/segmentation_manifest.csv',index=False)
ch[ch.in_segmentation].drop(columns=['fold_icdas','pipeline_pool']).to_csv(f'{O2}/children.csv',index=False)
print(sg.groupby('fold').agg(imgs=('image_id','size'),ninos=('child_id','nunique')))

## 3. README y checksums

In [ ]:
README_COMMON = """
## Source
All images derive from: Ahmed SMF, Ghori H, Khalid A, et al. *Annotated intraoral image dataset for dental caries detection*. Sci Data 2025;12:1297. Zenodo: https://doi.org/10.5281/zenodo.14827784 (CC BY 4.0). `ahmed_image` gives the path of the source photo inside `Dataset.zip`.

## Child identifier
Ahmed's file names identify a photo *submission*, not a child. Each child contributed one submission with cheek retractors (`..._01`) and one without (`..._00`), taken minutes apart. We paired them one-to-one (same prefix, consecutive number, <=30 min apart; 534 pairs). `child_id` is the resulting child; pilot sessions count as one child each. `children.csv` lists the submission keys of every child.

## Folds
5-fold cross-validation grouped by child (`fold` 0-4): all images of a child are in the same fold. ICDAS and segmentation share the same fold for the 65 children present in both datasets. Recommended use for test fold k: validation = fold (k+1) mod 5, training = remaining three folds.

## License
CC BY 4.0, as the source dataset. Please cite Ahmed et al. and this dataset.
"""
ICDAS = """# ICDAS tooth crops (curated subset of Ahmed et al.)
2,789 single-tooth crops from 514 photographs of 260 children, labelled with ICDAS II (0-6) and Merged-ICDAS (0 = sound, A = 1-2, B = 3-4, C = 5-6).

| Merged-ICDAS | 0 | A | B | C |
|---|---|---|---|---|
| Crops | 913 | 1,284 | 543 | 49 |

## Files
- `crops/` - one image per crop (`crop_id`).
- `icdas_manifest.csv` - `crop_id, crop_file, child_id, submission_key, ahmed_image, condition, view, x0, y0, x1, y1` (box in pixels of the original photo), `icdas, micdas, icdas_original, label_source, fold`.
- `children.csv` - one row per child.

## Curation
- Crops not traceable to a photo of the published Ahmed dataset were removed.
- Duplicate crops of the same tooth were detected (SIFT + pixel correlation) and reviewed; one crop per tooth was kept.
- Duplicate-group crops were relabelled independently by two dentists; disagreements were resolved by consensus. `icdas_original` keeps the first label; `label_source` says where the final label comes from.
""" + README_COMMON
SEG = """# FDI tooth segmentation (curated subset of Ahmed et al.)
630 intraoral photographs (126 submissions, 96 children) with per-pixel tooth masks labelled by FDI number.

## Files
- `images/`, `masks/` - 576 x 576 images and masks (`image_id`).
- `labelmap.txt` - class names and colours.
- `segmentation_manifest.csv` - `image_id, image_file, mask_file, child_id, submission_key, ahmed_image, condition, view, fold, a11...a23`. `a11...a23` is the affine matrix mapping original-photo pixel coordinates to the 576 x 576 image (includes letterbox, rotation and mirroring).
- `children.csv` - one row per child.

## Curation
- Every image was matched to its source photo in Ahmed et al. (thumbnail search + pixel correlation, SIFT fallback). Images from 17 submissions that could not be traced to the published dataset were removed.
""" + README_COMMON
open(f'{O1}/README.md','w').write(ICDAS); open(f'{O2}/README.md','w').write(SEG)
for O in (O1,O2):
    with open(f'{O}/checksums_md5.txt','w') as f:
        for p in sorted(glob.glob(f'{O}/**/*',recursive=True)):
            if os.path.isfile(p) and not p.endswith('checksums_md5.txt'):
                f.write(hashlib.md5(open(p,'rb').read()).hexdigest()+'  '+os.path.relpath(p,O)+'\n')
    print(O, sum(len(fs) for _,_,fs in os.walk(O)),'archivos')

## 4. Empaquetar (Kaggle solo pasa 500 archivos de un output a un dataset)

In [ ]:
W='/kaggle/working'
for O in (O1,O2):
    name=os.path.basename(O)
    shutil.make_archive(f'{W}/{name}','zip',root_dir=O)
    for f in os.listdir(O):
        if f.endswith('.csv') or f=='README.md':
            shutil.copy2(f'{O}/{f}', f"{W}/{name.split('_')[0]}_{f}" if f in ('children.csv','README.md') else f'{W}/{f}')
    shutil.rmtree(O)
for f in sorted(os.listdir(W)): print(f, round(os.path.getsize(f'{W}/{f}')/1e6,1),'MB')
